# 02 — Data Cleaning
This notebook cleans all raw CSV datasets and saves them to `../data/processed/`.

## Imports

In [ ]:
import pandas as pd
import os

---
## 1. Cleaning `nav_history.csv`

### 1.1 Load & inspect

In [ ]:
nav_history = pd.read_csv("../data/raw/02_nav_history.csv")
print(f"Shape: {nav_history.shape}")
print(f"Dtypes:\n{nav_history.dtypes}\n")
print(f"Null values:\n{nav_history.isnull().sum()}")

### 1.2 Convert date column to datetime

In [ ]:
nav_history["date"] = pd.to_datetime(nav_history["date"], format="%Y-%m-%d")
print("date dtype:", nav_history["date"].dtype)

### 1.3 Sort by amfi_code and date

In [ ]:
nav_history = nav_history.sort_values(by=["amfi_code", "date"])
print("First 5 rows of sorted nav_history:")
nav_history.head()

### 1.4 Forward-fill missing NAV values (holidays & weekends)

In [ ]:
nav_history["nav"] = nav_history.groupby("amfi_code")["nav"].ffill()
print("Missing NAV values after forward filling:", nav_history["nav"].isnull().sum())

### 1.5 Remove duplicates

In [ ]:
before = nav_history.duplicated().sum()
nav_history = nav_history.drop_duplicates()
print(f"Duplicate rows removed: {before}")

### 1.6 Validate NAV > 0

In [ ]:
invalid = (nav_history["nav"] <= 0).sum()
nav_history = nav_history[nav_history["nav"] > 0]
print(f"Rows with NAV <= 0 removed: {invalid}")

### 1.7 Save cleaned file

In [ ]:
nav_history.to_csv("../data/processed/cleaned_nav_history.csv", index=False)
print("Saved -> ../data/processed/cleaned_nav_history.csv")
print(f"Final shape: {nav_history.shape}")

---
## 2. Cleaning `investor_transactions.csv`

### 2.1 Load & inspect

In [ ]:
investor_transaction = pd.read_csv("../data/raw/08_investor_transactions.csv")
print(f"Shape: {investor_transaction.shape}")
print(f"Dtypes:\n{investor_transaction.dtypes}\n")
investor_transaction.head()

### 2.2 Validate amount > 0

In [ ]:
invalid = (investor_transaction["amount_inr"] <= 0).sum()
investor_transaction = investor_transaction[investor_transaction["amount_inr"] > 0]
print(f"Rows with amount <= 0 removed: {invalid}")

### 2.3 Standardize transaction types

In [ ]:
print("Before:", investor_transaction["transaction_type"].unique())

investor_transaction['transaction_type'] = (
    investor_transaction['transaction_type'].str.strip().str.lower()
)
mapping = {'sip': 'SIP', 'lumpsum': 'Lumpsum', 'redemption': 'Redemption'}
investor_transaction['transaction_type'] = investor_transaction['transaction_type'].map(mapping)

print("After: ", investor_transaction["transaction_type"].unique())

### 2.4 Convert transaction_date to datetime

In [ ]:
print("Before:", investor_transaction["transaction_date"].dtype)
investor_transaction["transaction_date"] = pd.to_datetime(
    investor_transaction["transaction_date"], format="%Y-%m-%d"
)
print("After: ", investor_transaction["transaction_date"].dtype)

### 2.5 Remove rows with invalid transaction types

In [ ]:
missing = investor_transaction["transaction_type"].isnull().sum()
print(f"Rows with missing transaction types: {missing}")
investor_transaction = investor_transaction.dropna(subset=['transaction_type'])

### 2.6 Check KYC status values

In [ ]:
print("Unique KYC status values:", investor_transaction["kyc_status"].unique())

### 2.7 Remove duplicates

In [ ]:
before = investor_transaction.duplicated().sum()
investor_transaction = investor_transaction.drop_duplicates()
print(f"Duplicate rows removed: {before}")

### 2.8 Save cleaned file

In [ ]:
investor_transaction.to_csv("../data/processed/cleaned_investor_transactions.csv", index=False)
print("Saved -> ../data/processed/cleaned_investor_transactions.csv")
print(f"Final shape: {investor_transaction.shape}")

---
## 3. Cleaning `scheme_performance.csv`

### 3.1 Load & inspect

In [ ]:
scheme_performance = pd.read_csv("../data/raw/07_scheme_performance.csv")
print(f"Shape: {scheme_performance.shape}")
print(f"Dtypes:\n{scheme_performance.dtypes}\n")
scheme_performance.head()

### 3.2 Convert return columns to numeric

In [ ]:
return_cols = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct']
for col in return_cols:
    scheme_performance[col] = pd.to_numeric(scheme_performance[col], errors='coerce')

print("Dtypes after conversion:")
print(scheme_performance[return_cols].dtypes)

### 3.3 Validate expense_ratio_pct range (0.1–2.5 %)

In [ ]:
range_check = (scheme_performance["expense_ratio_pct"] >= 0.1) & (scheme_performance["expense_ratio_pct"] <= 2.5)
print(f"Valid: {range_check.sum()}, Invalid: {(~range_check).sum()}")

### 3.4 Flag anomalous (negative) Sharpe ratios

In [ ]:
anomaly = (scheme_performance["sharpe_ratio"] < 0).sum()
print(f"Schemes with negative sharpe_ratio: {anomaly}")

### 3.5 Remove duplicates

In [ ]:
before = scheme_performance.duplicated().sum()
scheme_performance = scheme_performance.drop_duplicates()
print(f"Duplicate rows removed: {before}")

### 3.6 Save cleaned file

In [ ]:
scheme_performance.to_csv("../data/processed/cleaned_scheme_performance.csv", index=False)
print("Saved -> ../data/processed/cleaned_scheme_performance.csv")
print(f"Final shape: {scheme_performance.shape}")

---
## 4. Cleaning `fund_master.csv`

### 4.1 Load & inspect

In [ ]:
fund_master = pd.read_csv("../data/raw/01_fund_master.csv")
print(f"Shape: {fund_master.shape}")
print(f"Dtypes:\n{fund_master.dtypes}\n")
print(f"Null values:\n{fund_master.isnull().sum()}")

### 4.2 Convert launch_date & remove duplicates

In [ ]:
fund_master["launch_date"] = pd.to_datetime(fund_master["launch_date"], errors='coerce')

before = fund_master.duplicated().sum()
fund_master = fund_master.drop_duplicates()
print(f"Duplicates removed: {before}")

### 4.3 Validate numeric ranges & drop critical nulls

In [ ]:
# expense_ratio_pct: 0–3%, exit_load_pct: 0–5%
fund_master = fund_master[(fund_master["expense_ratio_pct"] >= 0) & (fund_master["expense_ratio_pct"] <= 3)]
fund_master = fund_master[(fund_master["exit_load_pct"] >= 0) & (fund_master["exit_load_pct"] <= 5)]

# Min amounts must be positive
fund_master = fund_master[fund_master["min_sip_amount"] > 0]
fund_master = fund_master[fund_master["min_lumpsum_amount"] > 0]

# Drop rows with critical nulls
fund_master = fund_master.dropna(subset=['amfi_code', 'fund_house', 'scheme_name', 'category'])
print(f"Final shape: {fund_master.shape}")

### 4.4 Save cleaned file

In [ ]:
fund_master.to_csv("../data/processed/cleaned_fund_master.csv", index=False)
print("Saved -> ../data/processed/cleaned_fund_master.csv")

---
## 5. Cleaning `aum_by_fund_house.csv`

### 5.1 Load & inspect

In [ ]:
aum_by_fund_house = pd.read_csv("../data/raw/03_aum_by_fund_house.csv")
print(f"Shape: {aum_by_fund_house.shape}")
print(f"Null values:\n{aum_by_fund_house.isnull().sum()}")

### 5.2 Convert date & remove duplicates

In [ ]:
aum_by_fund_house["date"] = pd.to_datetime(aum_by_fund_house["date"], errors='coerce')

before = aum_by_fund_house.duplicated().sum()
aum_by_fund_house = aum_by_fund_house.drop_duplicates()
print(f"Duplicates removed: {before}")

### 5.3 Validate, forward-fill & save

In [ ]:
aum_by_fund_house = aum_by_fund_house[aum_by_fund_house["aum_crore"] > 0]
aum_by_fund_house = aum_by_fund_house[aum_by_fund_house["num_schemes"] > 0]

aum_by_fund_house = aum_by_fund_house.sort_values(['fund_house', 'date'])
aum_by_fund_house["aum_lakh_crore"] = aum_by_fund_house.groupby("fund_house")["aum_lakh_crore"].ffill()
aum_by_fund_house = aum_by_fund_house.dropna(subset=['date', 'fund_house', 'aum_crore'])

print(f"Final shape: {aum_by_fund_house.shape}")
aum_by_fund_house.to_csv("../data/processed/cleaned_aum_by_fund_house.csv", index=False)
print("Saved -> ../data/processed/cleaned_aum_by_fund_house.csv")

---
## 6. Cleaning `monthly_sip_inflows.csv`

### 6.1 Load & inspect

In [ ]:
monthly_sip = pd.read_csv("../data/raw/04_monthly_sip_inflows.csv")
print(f"Shape: {monthly_sip.shape}")
print(f"Null values:\n{monthly_sip.isnull().sum()}")

### 6.2 Convert month & remove duplicates

In [ ]:
monthly_sip["month"] = pd.to_datetime(monthly_sip["month"], errors='coerce')

before = monthly_sip.duplicated().sum()
monthly_sip = monthly_sip.drop_duplicates()
print(f"Duplicates removed: {before}")

### 6.3 Validate numeric columns & save

In [ ]:
monthly_sip = monthly_sip[monthly_sip["sip_inflow_crore"] > 0]
monthly_sip = monthly_sip[monthly_sip["active_sip_accounts_crore"] > 0]
monthly_sip = monthly_sip[monthly_sip["new_sip_accounts_lakh"] > 0]
monthly_sip = monthly_sip[monthly_sip["sip_aum_lakh_crore"] > 0]
monthly_sip = monthly_sip[(monthly_sip["yoy_growth_pct"] >= -100) & (monthly_sip["yoy_growth_pct"] <= 100)]
monthly_sip = monthly_sip.dropna(subset=['month'])

print(f"Final shape: {monthly_sip.shape}")
monthly_sip.to_csv("../data/processed/cleaned_monthly_sip_inflows.csv", index=False)
print("Saved -> ../data/processed/cleaned_monthly_sip_inflows.csv")

---
## 7. Cleaning `category_inflows.csv`

### 7.1 Load & inspect

In [ ]:
category_inflows = pd.read_csv("../data/raw/05_category_inflows.csv")
print(f"Shape: {category_inflows.shape}")
print(f"Null values:\n{category_inflows.isnull().sum()}")

### 7.2 Clean & save

In [ ]:
category_inflows["month"] = pd.to_datetime(category_inflows["month"], errors='coerce')

before = category_inflows.duplicated().sum()
category_inflows = category_inflows.drop_duplicates()
print(f"Duplicates removed: {before}")

category_inflows = category_inflows[category_inflows["net_inflow_crore"].notna()]
category_inflows['category'] = category_inflows['category'].str.strip().str.title()
category_inflows = category_inflows.dropna(subset=['month', 'category'])

print(f"Final shape: {category_inflows.shape}")
category_inflows.to_csv("../data/processed/cleaned_category_inflows.csv", index=False)
print("Saved -> ../data/processed/cleaned_category_inflows.csv")

---
## 8. Cleaning `industry_folio_count.csv`

### 8.1 Load & inspect

In [ ]:
industry_folio = pd.read_csv("../data/raw/06_industry_folio_count.csv")
print(f"Shape: {industry_folio.shape}")
print(f"Null values:\n{industry_folio.isnull().sum()}")

### 8.2 Clean, validate & save

In [ ]:
industry_folio["month"] = pd.to_datetime(industry_folio["month"], errors='coerce')

before = industry_folio.duplicated().sum()
industry_folio = industry_folio.drop_duplicates()
print(f"Duplicates removed: {before}")

# Validate folio counts are positive
industry_folio = industry_folio[industry_folio["total_folios_crore"] > 0]
industry_folio = industry_folio[industry_folio["equity_folios_crore"] >= 0]
industry_folio = industry_folio[industry_folio["debt_folios_crore"] >= 0]
industry_folio = industry_folio[industry_folio["hybrid_folios_crore"] >= 0]
industry_folio = industry_folio[industry_folio["others_folios_crore"] >= 0]

# Validate component sum <= total
component_sum = (industry_folio['equity_folios_crore'] +
                 industry_folio['debt_folios_crore'] +
                 industry_folio['hybrid_folios_crore'] +
                 industry_folio['others_folios_crore'])
industry_folio = industry_folio[component_sum <= industry_folio['total_folios_crore'] * 1.01]
industry_folio = industry_folio.dropna(subset=['month'])

print(f"Final shape: {industry_folio.shape}")
industry_folio.to_csv("../data/processed/cleaned_industry_folio_count.csv", index=False)
print("Saved -> ../data/processed/cleaned_industry_folio_count.csv")

---
## 9. Cleaning `portfolio_holdings.csv`

### 9.1 Load & inspect

In [ ]:
portfolio_holdings = pd.read_csv("../data/raw/09_portfolio_holdings.csv")
print(f"Shape: {portfolio_holdings.shape}")
print(f"Null values:\n{portfolio_holdings.isnull().sum()}")

### 9.2 Clean, validate & save

In [ ]:
portfolio_holdings["portfolio_date"] = pd.to_datetime(portfolio_holdings["portfolio_date"], errors='coerce')

before = portfolio_holdings.duplicated().sum()
portfolio_holdings = portfolio_holdings.drop_duplicates()
print(f"Duplicates removed: {before}")

# Validate ranges
portfolio_holdings = portfolio_holdings[(portfolio_holdings["weight_pct"] >= 0) & (portfolio_holdings["weight_pct"] <= 100)]
portfolio_holdings = portfolio_holdings[portfolio_holdings["current_price_inr"] > 0]
portfolio_holdings = portfolio_holdings[portfolio_holdings["market_value_cr"] > 0]

# Standardize sector names
portfolio_holdings['sector'] = portfolio_holdings['sector'].str.strip().str.title()
portfolio_holdings = portfolio_holdings.dropna(subset=['amfi_code', 'stock_symbol', 'portfolio_date'])

print(f"Final shape: {portfolio_holdings.shape}")
portfolio_holdings.to_csv("../data/processed/cleaned_portfolio_holdings.csv", index=False)
print("Saved -> ../data/processed/cleaned_portfolio_holdings.csv")

---
## 10. Cleaning `benchmark_indices.csv`

### 10.1 Load & inspect

In [ ]:
benchmark_indices = pd.read_csv("../data/raw/10_benchmark_indices.csv")
print(f"Shape: {benchmark_indices.shape}")
print(f"Null values:\n{benchmark_indices.isnull().sum()}")

### 10.2 Clean, validate & save

In [ ]:
benchmark_indices["date"] = pd.to_datetime(benchmark_indices["date"], errors='coerce')

before = benchmark_indices.duplicated().sum()
benchmark_indices = benchmark_indices.drop_duplicates()
print(f"Duplicates removed: {before}")

benchmark_indices = benchmark_indices[benchmark_indices["close_value"] > 0]
benchmark_indices['index_name'] = benchmark_indices['index_name'].str.strip().str.upper()
benchmark_indices = benchmark_indices.sort_values(['index_name', 'date'])
benchmark_indices['close_value'] = benchmark_indices.groupby('index_name')['close_value'].ffill()
benchmark_indices = benchmark_indices.dropna(subset=['date', 'index_name', 'close_value'])

print(f"Final shape: {benchmark_indices.shape}")
benchmark_indices.to_csv("../data/processed/cleaned_benchmark_indices.csv", index=False)
print("Saved -> ../data/processed/cleaned_benchmark_indices.csv")

---
## Cleaning Summary — All Datasets

In [ ]:
processed_dir = "../data/processed"
cleaned_files = sorted([f for f in os.listdir(processed_dir) if f.startswith("cleaned_")])

summary_data = []
for file in cleaned_files:
    df = pd.read_csv(os.path.join(processed_dir, file))
    summary_data.append({
        'File': file,
        'Rows': df.shape[0],
        'Columns': df.shape[1],
        'Null Count': df.isnull().sum().sum()
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

print(f"\nTotal cleaned datasets: {len(cleaned_files)}")
print("\nAll datasets have been:")
print("  ✓ Handled missing values (nulls)")
print("  ✓ Removed duplicate rows")
print("  ✓ Validated data types")
print("  ✓ Checked numeric ranges")
print("  ✓ Standardized categorical values")
print("\nCleaned files saved in: ../data/processed/")